In [42]:
# ============================================================
# CORREGIR CONFLICTO PROJ / POSTGIS / RASTERIO
# ============================================================

import os
import pyproj

PROJ_DIR = pyproj.datadir.get_data_dir()

os.environ["PROJ_LIB"] = PROJ_DIR
os.environ["PROJ_DATA"] = PROJ_DIR

print("PROJ_DIR usado por Python:")
print(PROJ_DIR)

PROJ_DIR usado por Python:
c:\Users\JATL2\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyproj\proj_dir\share\proj


In [39]:
import ee
import geemap
from IPython.display import display

PROJECT_ID = "ee-juatorreslo"

ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine inicializado correctamente.")

Google Earth Engine inicializado correctamente.


In [3]:
CONFIG = {
    "fecha_inicio": "2015-01-01",
    "fecha_fin": "2024-12-31",

    # Método mensual:
    # "median" = más conservador
    # "p10"    = recomendado para captar agua temporal
    # "min"    = muy sensible, puede sobredetectar
    "metodo_mensual": "p10",

    # Umbrales SAR
    # Si detecta muy poco, prueba -17
    # Si detecta demasiado, prueba -19
    "umbral_vv": -18,
    "umbral_vh": -23,

    # False = solo VV
    # True  = VV y VH simultáneamente
    "usar_vh": False,

    # Pendiente máxima en grados
    "pendiente_max": 5,

    # Escala para exportar raster
    "escala": 30,

    # Escala para calcular áreas
    # 250 = rápido
    # 100 = más detallado
    # 30  = más preciso, pero puede tardar mucho
    "escala_area": 250,

    # Mínimo de meses detectados como inundados
    "min_meses_inundado": 2,

    # Limpieza espacial
    "min_pixeles_conectados": 8,

    # Carpeta de Google Drive
    "carpeta_drive": "GEE_Inundaciones_Meta",

    # Mes específico para visualizar/exportar
    "mes_visualizar": "2024-05",
}

metodos_validos = ["median", "min", "p10"]

if CONFIG["metodo_mensual"] not in metodos_validos:
    raise ValueError(
        f"metodo_mensual inválido: {CONFIG['metodo_mensual']}. "
        f"Usa uno de estos: {metodos_validos}"
    )

print("Configuración cargada.")

Configuración cargada.


In [4]:
# ============================================================
# ÁREA DE ESTUDIO
# ============================================================

AOI_MODO = "gaul_meta"

# Opciones:
# "gaul_meta" -> Departamento del Meta desde GAUL
# "asset"     -> AOI subido como asset de GEE
# "shp"       -> shapefile local
# "geojson"   -> GeoJSON local


if AOI_MODO == "gaul_meta":

    zona = (
        ee.FeatureCollection("FAO/GAUL/2015/level1")
        .filter(ee.Filter.eq("ADM0_NAME", "Colombia"))
        .filter(ee.Filter.eq("ADM1_NAME", "Meta"))
    )

    geom = zona.geometry()


elif AOI_MODO == "asset":

    # Cambia esta ruta por la ruta real de tu asset
    AOI_ASSET = "projects/ee-juatorreslo/assets/aoi_inundaciones"

    zona = ee.FeatureCollection(AOI_ASSET)
    geom = zona.geometry()


elif AOI_MODO == "shp":

    # Cambia esta ruta por tu shapefile
    RUTA_SHP = r"D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\aoi.shp"

    zona = geemap.shp_to_ee(RUTA_SHP)
    geom = zona.geometry()


elif AOI_MODO == "geojson":

    # Cambia esta ruta por tu GeoJSON
    RUTA_GEOJSON = r"D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\aoi.geojson"

    zona = geemap.geojson_to_ee(RUTA_GEOJSON)
    geom = zona.geometry()


else:
    raise ValueError("AOI_MODO no válido.")


print("Área de estudio cargada correctamente.")

Área de estudio cargada correctamente.


In [5]:
Map = geemap.Map()
Map.centerObject(zona, 8)
Map.addLayer(zona, {"color": "red"}, "Área de estudio")
display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [6]:
bandas_s1 = ["VV", "VH"] if CONFIG["usar_vh"] else ["VV"]

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(geom)
    .filterDate(CONFIG["fecha_inicio"], CONFIG["fecha_fin"])
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
    .select(bandas_s1)
)

print("Número de imágenes Sentinel-1 originales:", s1.size().getInfo())

Número de imágenes Sentinel-1 originales: 2394


In [7]:
# Agua permanente histórica JRC
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

jrc_mask = jrc.select("seasonality").mask()

agua_permanente = (
    jrc.select("seasonality")
    .gte(10)
    .updateMask(jrc_mask)
    .unmask(0)
    .clip(geom)
)

# Pendiente SRTM
dem = ee.Image("USGS/SRTMGL1_003").clip(geom)

pendiente = ee.Terrain.slope(dem).clip(geom)

print("Capas auxiliares creadas.")

Capas auxiliares creadas.


In [8]:
Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    agua_permanente.updateMask(agua_permanente),
    {"palette": ["0000ff"]},
    "Agua permanente JRC"
)

Map.addLayer(
    pendiente,
    {"min": 0, "max": 30},
    "Pendiente SRTM"
)

display(Map)

Map(center=[3.3188729203771468, -72.9668566980664], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
fecha_inicio_ee = ee.Date(CONFIG["fecha_inicio"])
fecha_fin_ee = ee.Date(CONFIG["fecha_fin"]).advance(1, "day")

n_meses = fecha_fin_ee.difference(fecha_inicio_ee, "month").ceil()
lista_meses = ee.List.sequence(0, n_meses.subtract(1))


def crear_feature_mes(m):
    m = ee.Number(m)

    inicio_mes = fecha_inicio_ee.advance(m, "month")
    fin_mes = inicio_mes.advance(1, "month")

    col_mes = s1.filterDate(inicio_mes, fin_mes)
    n_img_mes = col_mes.size()

    return ee.Feature(
        None,
        {
            "m": m,
            "inicio_millis": inicio_mes.millis(),
            "periodo": inicio_mes.format("YYYY-MM"),
            "year": inicio_mes.get("year"),
            "mes": inicio_mes.get("month"),
            "n_imagenes_mes": n_img_mes,
        },
    )


meses_fc = (
    ee.FeatureCollection(lista_meses.map(crear_feature_mes))
    .filter(ee.Filter.gt("n_imagenes_mes", 0))
)

meses_lista = meses_fc.toList(meses_fc.size())


def crear_compuesto_mensual(f):
    f = ee.Feature(f)

    inicio_mes = ee.Date(f.get("inicio_millis"))
    fin_mes = inicio_mes.advance(1, "month")

    col_mes = s1.filterDate(inicio_mes, fin_mes)

    if CONFIG["metodo_mensual"] == "median":
        reducido = col_mes.median()

    elif CONFIG["metodo_mensual"] == "min":
        reducido = col_mes.min()

    elif CONFIG["metodo_mensual"] == "p10":
        reducido = col_mes.reduce(ee.Reducer.percentile([10]))

    else:
        reducido = col_mes.median()

    bandas = ["VV", "VH"] if CONFIG["usar_vh"] else ["VV"]

    reducido = reducido.rename(bandas)

    vv_suavizado = (
        reducido.select("VV")
        .focal_mean(radius=50, kernelType="circle", units="meters")
        .rename("VV_suavizado")
    )

    compuesto = vv_suavizado

    if CONFIG["usar_vh"]:
        vh_suavizado = (
            reducido.select("VH")
            .focal_mean(radius=50, kernelType="circle", units="meters")
            .rename("VH_suavizado")
        )

        vv_menos_vh = (
            vv_suavizado
            .subtract(vh_suavizado)
            .rename("VV_menos_VH")
        )

        compuesto = (
            vv_suavizado
            .addBands(vh_suavizado)
            .addBands(vv_menos_vh)
        )

    return (
        compuesto
        .clip(geom)
        .set("system:time_start", inicio_mes.millis())
        .set("periodo", f.get("periodo"))
        .set("year", f.get("year"))
        .set("mes", f.get("mes"))
        .set("n_imagenes_mes", f.get("n_imagenes_mes"))
        .set("metodo_mensual", CONFIG["metodo_mensual"])
    )


s1_mensual = ee.ImageCollection.fromImages(
    meses_lista.map(crear_compuesto_mensual)
)

print("Número de imágenes mensuales:", s1_mensual.size().getInfo())

Número de imágenes mensuales: 108


In [11]:
compuesto_mes = ee.Image(
    s1_mensual
    .filter(ee.Filter.eq("periodo", CONFIG["mes_visualizar"]))
    .first()
)

Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    compuesto_mes.select("VV_suavizado"),
    {"min": -25, "max": 0},
    f"Compuesto VV mensual {CONFIG['mes_visualizar']}"
)

display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [12]:
def detectar_inundacion(img):
    vv = img.select("VV_suavizado")

    # VV bajo = posible agua
    # Se eliminan valores extremadamente bajos para reducir sombras/artefactos
    agua_sar = (
        vv.lt(CONFIG["umbral_vv"])
        .And(vv.gt(-30))
    )

    if CONFIG["usar_vh"]:
        agua_sar = agua_sar.And(
            img.select("VH_suavizado").lt(CONFIG["umbral_vh"])
        )

    inundacion = (
        agua_sar
        .And(agua_permanente.Not())
        .And(pendiente.lt(CONFIG["pendiente_max"]))
        .rename("inundacion")
        .clip(geom)
    )

    # Limpieza espacial
    inundacion_mascara = inundacion.selfMask()

    pixeles_conectados = inundacion_mascara.connectedPixelCount(
        100,
        True
    )

    inundacion = (
        inundacion_mascara
        .updateMask(
            pixeles_conectados.gte(CONFIG["min_pixeles_conectados"])
        )
        .unmask(0)
        .rename("inundacion")
        .clip(geom)
    )

    return inundacion.copyProperties(
        img,
        [
            "system:time_start",
            "periodo",
            "year",
            "mes",
            "n_imagenes_mes",
            "metodo_mensual",
        ],
    )


inundaciones = s1_mensual.map(detectar_inundacion)

print("Número de máscaras mensuales:", inundaciones.size().getInfo())

Número de máscaras mensuales: 108


In [13]:
inundacion_mes = ee.Image(
    inundaciones
    .filter(ee.Filter.eq("periodo", CONFIG["mes_visualizar"]))
    .first()
)

Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    compuesto_mes.select("VV_suavizado"),
    {"min": -25, "max": 0},
    f"Compuesto VV mensual {CONFIG['mes_visualizar']}"
)

Map.addLayer(
    inundacion_mes.updateMask(inundacion_mes),
    {"palette": ["00ffff"]},
    f"Posible inundación mensual {CONFIG['mes_visualizar']}"
)

display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [15]:
total_meses = inundaciones.size()

frecuencia_absoluta = (
    inundaciones
    .sum()
    .rename("frecuencia_absoluta_meses")
    .clip(geom)
)

frecuencia_relativa = (
    frecuencia_absoluta
    .divide(ee.Image.constant(total_meses))
    .multiply(100)
    .rename("frecuencia_relativa_pct")
    .clip(geom)
)

print("Frecuencia absoluta y relativa calculadas.")

Frecuencia absoluta y relativa calculadas.


In [16]:
recurrencia = (
    ee.Image(0)
    .where(
        frecuencia_relativa.gt(0).And(frecuencia_relativa.lte(5)),
        1
    )
    .where(
        frecuencia_relativa.gt(5).And(frecuencia_relativa.lte(15)),
        2
    )
    .where(
        frecuencia_relativa.gt(15),
        3
    )
    .rename("recurrencia_inundacion")
    .clip(geom)
)

print("Clasificación de recurrencia creada.")

Clasificación de recurrencia creada.


In [17]:
Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    frecuencia_relativa,
    {
        "min": 0,
        "max": 25,
        "palette": [
            "ffffff",
            "d0e1f2",
            "74a9cf",
            "2b8cbe",
            "045a8d",
        ],
    },
    "Frecuencia relativa de inundación (%)"
)

Map.addLayer(
    recurrencia.updateMask(recurrencia),
    {
        "min": 1,
        "max": 3,
        "palette": [
            "ffffb2",
            "fecc5c",
            "e31a1c",
        ],
    },
    "Recurrencia de inundación"
)

display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [18]:
years = list(range(2015, 2025))

imagenes_anuales = []
nombres_bandas_anuales = []

for year in years:
    col_anual = inundaciones.filter(ee.Filter.eq("year", year))
    n_meses_year = col_anual.size()

    nombre_banda = f"frec_{year}"
    nombres_bandas_anuales.append(nombre_banda)

    imagen_anual = ee.Image(
        ee.Algorithms.If(
            n_meses_year.gt(0),
            col_anual
            .sum()
            .divide(ee.Image.constant(n_meses_year))
            .multiply(100)
            .rename(nombre_banda)
            .clip(geom),
            ee.Image.constant(0)
            .rename(nombre_banda)
            .clip(geom),
        )
    ).set("year", year)

    imagenes_anuales.append(imagen_anual)


frecuencia_anual_bandas = (
    ee.Image.cat(*imagenes_anuales)
    .rename(nombres_bandas_anuales)
    .clip(geom)
)

print("Bandas frecuencia anual:", frecuencia_anual_bandas.bandNames().getInfo())

Bandas frecuencia anual: ['frec_2015', 'frec_2016', 'frec_2017', 'frec_2018', 'frec_2019', 'frec_2020', 'frec_2021', 'frec_2022', 'frec_2023', 'frec_2024']


In [19]:
YEAR_VISUALIZAR = 2024

Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    frecuencia_anual_bandas.select(f"frec_{YEAR_VISUALIZAR}"),
    {
        "min": 0,
        "max": 25,
        "palette": [
            "ffffff",
            "d0e1f2",
            "74a9cf",
            "2b8cbe",
            "045a8d",
        ],
    },
    f"Frecuencia de inundación {YEAR_VISUALIZAR}"
)

display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [20]:
inundacion_recurrente_periodo = (
    frecuencia_absoluta
    .gte(CONFIG["min_meses_inundado"])
    .rename("inundacion_recurrente_periodo")
    .clip(geom)
)

print("Zona inundable recurrente creada.")

Zona inundable recurrente creada.


In [21]:
Map = geemap.Map()
Map.centerObject(zona, 8)

Map.addLayer(zona, {"color": "red"}, "Área de estudio")

Map.addLayer(
    compuesto_mes.select("VV_suavizado"),
    {"min": -25, "max": 0},
    f"Compuesto VV mensual {CONFIG['mes_visualizar']}"
)

Map.addLayer(
    inundacion_recurrente_periodo.updateMask(inundacion_recurrente_periodo),
    {"palette": ["0000ff"]},
    f"Zona inundable recurrente mínimo {CONFIG['min_meses_inundado']} meses"
)

display(Map)

Map(center=[3.3188729203771605, -72.96685669806644], controls=(WidgetControl(options=['position', 'transparent…

In [22]:
def calcular_area_ha(img):
    area = (
        ee.Image.pixelArea()
        .divide(10000)
        .updateMask(img.eq(1))
        .rename("area_ha")
    )

    stats = area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geom,
        scale=CONFIG["escala_area"],
        maxPixels=1e13,
        tileScale=8,
        bestEffort=True
    )

    return ee.Number(stats.get("area_ha"))


print("Función de área creada.")

Función de área creada.


In [23]:
def crear_feature_area_mensual(img):
    img = ee.Image(img)

    area_ha = calcular_area_ha(img)
    area_km2 = area_ha.divide(100)

    return ee.Feature(
        None,
        {
            "periodo": img.get("periodo"),
            "year": img.get("year"),
            "mes": img.get("mes"),
            "n_imagenes_mes": img.get("n_imagenes_mes"),
            "area_inundada_ha": area_ha.round(),
            "area_inundada_km2": area_km2.round(),
            "escala_area_m": CONFIG["escala_area"],
        },
    )


lista_inundaciones = inundaciones.toList(inundaciones.size())

tabla_area_mensual = ee.FeatureCollection(
    lista_inundaciones.map(crear_feature_area_mensual)
)

print("Tabla de área mensual creada.")

Tabla de área mensual creada.


In [24]:
features_anuales = []

for year in years:
    inundacion_anual = (
        inundaciones
        .filter(ee.Filter.eq("year", year))
        .max()
        .rename("inundacion_anual")
        .clip(geom)
    )

    area_ha = calcular_area_ha(inundacion_anual)
    area_km2 = area_ha.divide(100)

    feature = ee.Feature(
        None,
        {
            "year": year,
            "area_inundada_ha": area_ha.round(),
            "area_inundada_km2": area_km2.round(),
            "escala_area_m": CONFIG["escala_area"],
        },
    )

    features_anuales.append(feature)


tabla_area_anual = ee.FeatureCollection(features_anuales)

print("Tabla de área anual creada.")

Tabla de área anual creada.


In [25]:
area_recurrente_ha = calcular_area_ha(inundacion_recurrente_periodo)
area_recurrente_km2 = area_recurrente_ha.divide(100)

tabla_area_recurrente_periodo = ee.FeatureCollection(
    [
        ee.Feature(
            None,
            {
                "periodo": "2015-2024",
                "criterio": f"minimo_{CONFIG['min_meses_inundado']}_meses",
                "area_inundable_recurrente_ha": area_recurrente_ha.round(),
                "area_inundable_recurrente_km2": area_recurrente_km2.round(),
                "escala_area_m": CONFIG["escala_area"],
            },
        )
    ]
)

print("Tabla de área recurrente del periodo creada.")

Tabla de área recurrente del periodo creada.


In [26]:
categorias = {
    1: "Baja",
    2: "Media",
    3: "Alta",
}

features_clases = []

for clase, categoria in categorias.items():
    mascara_clase = (
        recurrencia
        .eq(clase)
        .rename("clase")
        .clip(geom)
    )

    area_ha = calcular_area_ha(mascara_clase)
    area_km2 = area_ha.divide(100)

    feature = ee.Feature(
        None,
        {
            "clase_recurrencia": clase,
            "categoria": categoria,
            "area_ha": area_ha.round(),
            "area_km2": area_km2.round(),
            "escala_area_m": CONFIG["escala_area"],
        },
    )

    features_clases.append(feature)


tabla_area_por_clase = ee.FeatureCollection(features_clases)

print("Tabla de área por clase de recurrencia creada.")

Tabla de área por clase de recurrencia creada.


In [28]:
import os

SALIDA_LOCAL = r"D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee"

os.makedirs(SALIDA_LOCAL, exist_ok=True)

print("Carpeta de salida:", SALIDA_LOCAL)

Carpeta de salida: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee


In [29]:
tabla_area_mensual
tabla_area_anual
tabla_area_recurrente_periodo
tabla_area_por_clase

In [31]:
# ============================================================
# EXPORTAR TABLAS LOCALMENTE
# ============================================================

import os
import geemap

SALIDA_LOCAL = r"D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee"

os.makedirs(SALIDA_LOCAL, exist_ok=True)

print("Carpeta de salida:", SALIDA_LOCAL)


def exportar_csv_local(fc, nombre_archivo, columnas):
    """
    Exporta una FeatureCollection de Earth Engine como CSV local,
    seleccionando previamente solo las columnas necesarias.
    """
    fc_limpia = fc.select(columnas)

    ruta_salida = os.path.join(SALIDA_LOCAL, nombre_archivo)

    geemap.ee_to_csv(
        fc_limpia,
        filename=ruta_salida
    )

    print("CSV exportado:", ruta_salida)


# ------------------------------------------------------------
# 1. Área inundada mensual
# ------------------------------------------------------------

exportar_csv_local(
    tabla_area_mensual,
    "AOI_area_inundada_mensual_2015_2024.csv",
    [
        "periodo",
        "year",
        "mes",
        "n_imagenes_mes",
        "area_inundada_ha",
        "area_inundada_km2",
        "escala_area_m",
    ]
)


# ------------------------------------------------------------
# 2. Área inundada anual
# ------------------------------------------------------------

exportar_csv_local(
    tabla_area_anual,
    "AOI_area_inundada_anual_2015_2024.csv",
    [
        "year",
        "area_inundada_ha",
        "area_inundada_km2",
        "escala_area_m",
    ]
)


# ------------------------------------------------------------
# 3. Área inundable recurrente 2015-2024
# ------------------------------------------------------------

exportar_csv_local(
    tabla_area_recurrente_periodo,
    "AOI_area_inundable_recurrente_2015_2024.csv",
    [
        "periodo",
        "criterio",
        "area_inundable_recurrente_ha",
        "area_inundable_recurrente_km2",
        "escala_area_m",
    ]
)


# ------------------------------------------------------------
# 4. Área por clase de recurrencia
# ------------------------------------------------------------

exportar_csv_local(
    tabla_area_por_clase,
    "AOI_area_por_clase_recurrencia_2015_2024.csv",
    [
        "clase_recurrencia",
        "categoria",
        "area_ha",
        "area_km2",
        "escala_area_m",
    ]
)

print("Proceso de exportación local terminado.")

Carpeta de salida: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee
Too many concurrent aggregations.
CSV exportado: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee\AOI_area_inundada_mensual_2015_2024.csv
CSV exportado: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee\AOI_area_inundada_anual_2015_2024.csv
CSV exportado: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee\AOI_area_inundable_recurrente_2015_2024.csv
CSV exportado: D:\MAESTRIA\PROGRAMACION_SIG\juatorreslo\PROYECTO_FINAL\VISUAL\salidas_gee\AOI_area_por_clase_recurrencia_2015_2024.csv
Proceso de exportación local terminado.


In [32]:
frecuencia_relativa
frecuencia_absoluta
recurrencia
frecuencia_anual_bandas
inundacion_recurrente_periodo
inundacion_mes
tabla_area_mensual
tabla_area_anual
tabla_area_recurrente_periodo
tabla_area_por_clase
geom
CONFIG

{'fecha_inicio': '2015-01-01',
 'fecha_fin': '2024-12-31',
 'metodo_mensual': 'p10',
 'umbral_vv': -18,
 'umbral_vh': -23,
 'usar_vh': False,
 'pendiente_max': 5,
 'escala': 30,
 'escala_area': 250,
 'min_meses_inundado': 2,
 'min_pixeles_conectados': 8,
 'carpeta_drive': 'GEE_Inundaciones_Meta',
 'mes_visualizar': '2024-05'}

In [51]:
# ============================================================
# EXPORTACIONES A GOOGLE DRIVE
# Rasters GeoTIFF + tablas CSV
# ============================================================

import ee

CARPETA_DRIVE = CONFIG["carpeta_drive"]

print("Carpeta de Google Drive:", CARPETA_DRIVE)


# ============================================================
# FUNCIONES DE EXPORTACIÓN
# ============================================================

def exportar_raster_drive(
    imagen,
    nombre,
    escala=None,
    region=None,
    tipo="float"
):
    """
    Exporta un ee.Image a Google Drive como GeoTIFF.
    """

    if escala is None:
        escala = CONFIG["escala"]

    if region is None:
        region = geom

    # Ajuste de tipo de dato
    if tipo == "float":
        imagen_exportar = imagen.toFloat()
    elif tipo == "int16":
        imagen_exportar = imagen.toInt16()
    elif tipo == "byte":
        imagen_exportar = imagen.toByte()
    else:
        imagen_exportar = imagen

    task = ee.batch.Export.image.toDrive(
        image=imagen_exportar.clip(region),
        description=nombre,
        folder=CARPETA_DRIVE,
        fileNamePrefix=nombre,
        region=region,
        scale=escala,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={
            "cloudOptimized": True
        },
        skipEmptyTiles=True,
        fileDimensions=8192
    )

    task.start()

    print("Exportación raster iniciada:", nombre)


def exportar_tabla_drive(
    coleccion,
    nombre,
    columnas
):
    """
    Exporta una ee.FeatureCollection a Google Drive como CSV.
    """

    task = ee.batch.Export.table.toDrive(
        collection=coleccion,
        description=nombre,
        folder=CARPETA_DRIVE,
        fileNamePrefix=nombre,
        fileFormat="CSV",
        selectors=columnas
    )

    task.start()

    print("Exportación tabla iniciada:", nombre)

Carpeta de Google Drive: GEE_Inundaciones_Meta


In [52]:
# ============================================================
# RASTERS PRINCIPALES
# ============================================================

exportar_raster_drive(
    frecuencia_relativa,
    "AOI_frecuencia_relativa_inundacion_mensual_2015_2024",
    escala=CONFIG["escala"],
    tipo="float"
)

exportar_raster_drive(
    frecuencia_absoluta,
    "AOI_frecuencia_absoluta_inundacion_mensual_2015_2024",
    escala=CONFIG["escala"],
    tipo="int16"
)

exportar_raster_drive(
    recurrencia,
    "AOI_recurrencia_inundacion_mensual_2015_2024",
    escala=CONFIG["escala"],
    tipo="byte"
)

exportar_raster_drive(
    inundacion_recurrente_periodo,
    f"AOI_zona_inundable_recurrente_min_{CONFIG['min_meses_inundado']}_meses_2015_2024",
    escala=CONFIG["escala"],
    tipo="byte"
)

exportar_raster_drive(
    inundacion_mes,
    f"AOI_inundacion_mensual_{CONFIG['mes_visualizar']}",
    escala=CONFIG["escala"],
    tipo="byte"
)

Exportación raster iniciada: AOI_frecuencia_relativa_inundacion_mensual_2015_2024
Exportación raster iniciada: AOI_frecuencia_absoluta_inundacion_mensual_2015_2024
Exportación raster iniciada: AOI_recurrencia_inundacion_mensual_2015_2024
Exportación raster iniciada: AOI_zona_inundable_recurrente_min_2_meses_2015_2024
Exportación raster iniciada: AOI_inundacion_mensual_2024-05


In [53]:
# ============================================================
# FRECUENCIA ANUAL MULTIBANDA
# Una banda por año: frec_2015, frec_2016, ..., frec_2024
# ============================================================

exportar_raster_drive(
    frecuencia_anual_bandas,
    "AOI_frecuencia_anual_inundacion_mensual_2015_2024",
    escala=CONFIG["escala"],
    tipo="float"
)

Exportación raster iniciada: AOI_frecuencia_anual_inundacion_mensual_2015_2024


In [54]:
# ============================================================
# FRECUENCIA ANUAL POR AÑO
# ============================================================

for year in range(2015, 2025):
    banda = f"frec_{year}"

    exportar_raster_drive(
        frecuencia_anual_bandas.select(banda),
        f"AOI_frecuencia_inundacion_{year}",
        escala=CONFIG["escala"],
        tipo="float"
    )

Exportación raster iniciada: AOI_frecuencia_inundacion_2015
Exportación raster iniciada: AOI_frecuencia_inundacion_2016
Exportación raster iniciada: AOI_frecuencia_inundacion_2017
Exportación raster iniciada: AOI_frecuencia_inundacion_2018
Exportación raster iniciada: AOI_frecuencia_inundacion_2019
Exportación raster iniciada: AOI_frecuencia_inundacion_2020
Exportación raster iniciada: AOI_frecuencia_inundacion_2021
Exportación raster iniciada: AOI_frecuencia_inundacion_2022
Exportación raster iniciada: AOI_frecuencia_inundacion_2023
Exportación raster iniciada: AOI_frecuencia_inundacion_2024


In [49]:
# ============================================================
# TABLAS CSV
# ============================================================

exportar_tabla_drive(
    tabla_area_mensual,
    "AOI_area_inundada_mensual_2015_2024",
    [
        "periodo",
        "year",
        "mes",
        "n_imagenes_mes",
        "area_inundada_ha",
        "area_inundada_km2",
        "escala_area_m",
    ]
)

exportar_tabla_drive(
    tabla_area_anual,
    "AOI_area_inundada_anual_2015_2024",
    [
        "year",
        "area_inundada_ha",
        "area_inundada_km2",
        "escala_area_m",
    ]
)

exportar_tabla_drive(
    tabla_area_recurrente_periodo,
    "AOI_area_inundable_recurrente_2015_2024",
    [
        "periodo",
        "criterio",
        "area_inundable_recurrente_ha",
        "area_inundable_recurrente_km2",
        "escala_area_m",
    ]
)

exportar_tabla_drive(
    tabla_area_por_clase,
    "AOI_area_por_clase_recurrencia_2015_2024",
    [
        "clase_recurrencia",
        "categoria",
        "area_ha",
        "area_km2",
        "escala_area_m",
    ]
)

Exportación tabla iniciada: AOI_area_inundada_mensual_2015_2024
Exportación tabla iniciada: AOI_area_inundada_anual_2015_2024
Exportación tabla iniciada: AOI_area_inundable_recurrente_2015_2024
Exportación tabla iniciada: AOI_area_por_clase_recurrencia_2015_2024
